In [9]:
# Define parameters
keyword = '/g/11c265kd70'
gprop = 'youtube'
geo = 'IN'
timeframe = '2025-06-15 2025-12-29'
params = {
    "api_key" : "b9470e54fd35b3f2174c686b526660af023bbcb1521aa1211194bef5a5a663c4",
    "engine": "google_trends",
    "q": keyword,
    "geo": geo,
    "date": timeframe,
    "data_type": "TIMESERIES",
    "tz": "330",
    "gprop": gprop if gprop != "web" else ""
}

In [10]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher
fetcher = SerpApiTrendsFetcher(geo="IN")
# start = 
# end = 
try:
    # fetcher.initialize()
    data = fetcher.fetch_chunk(keyword, '2025-06-15', '2025-12-29')
    print(f"fetched data:{data}")
except Exception as e:
    print(f"error:{e}")


error:SerpApiKeyManager not initialized


In [ ]:
data = fetcher.fetch_chunk(
    keyword="/g/11c265kd70",
    start="2025-06-01",
    end="2025-12-31",
    gprop="youtube"
)

In [ ]:
from datetime import datetime, timedelta
from statistics import mean
import gtrends.db.utils as db_utils
from gtrends.db.keywords import GoogleTrendsRepository

class GoogleTrendSyncer:
    """
    Handles normalization and continuity of Google Trends time series
    across overlapping fetch windows.
     - Feeds it into the database while also updating the past entries
    """

    DATE_FMT = "%d-%m-%Y"

    def __init__(self):
        self.gtr = GoogleTrendsRepository()
    
    def _find_overlap(self, past, new):
        past_map = {p["date"]: p["value"] for p in past}
        overlap = []

        for row in new:
            if row["date"] in past_map:
                overlap.append((past_map[row["date"]], row["value"]))

        return overlap
    
    def compute_scaling_factor(self, overlap):
        """
        Scaling factor = mean(past_values) / mean(new_values)
        """
        if len(overlap) < 1:
            return 1.0  # insufficient overlap → no scaling

        past_vals = [p for p, _ in overlap]
        new_vals = [n for _, n in overlap]

        if mean(new_vals) == 0:
            return 1.0

        return mean(past_vals) / mean(new_vals)
    
    def normalize_new_data(self, new_data, scale):
        normalized = []

        for row in new_data:
            normalized.append({
                "date": row["date"],
                "value": round(row["value"] * scale, 2)
            })

        return normalized
    
    def merge_timeseries(self, past, normalized_new):
        merged = {row["date"]: row["value"] for row in past}

        for row in normalized_new:
            merged[row["date"]] = row["value"]

        return [
            {"date": d, "value": v}
            for d, v in sorted(merged.items(), key=lambda x: datetime.strptime(x[0], self.DATE_FMT))
        ]
    
    def upsert_timeseries(self, keyword, geo, data):
        query = """
        INSERT INTO google_trends_timeseries (keyword, geo, date, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value)
        """

        params = [
            (
                keyword,
                geo,
                datetime.strptime(row["date"], self.DATE_FMT),
                row["value"]
            )
            for row in data
        ]

        self.db.executemany(query, params)
        self.db.commit()

    def generate_windows(self, start, end, window_days = 240, overlap_days = 30):
        w_end = end

        windows = []
        while w_end > start:
            w_start = max(w_end - timedelta(days = window_days), start) # enforces 30 days overlap
            windows.append((w_start.isoformat(), w_end.isoformat()))
            w_end = w_end - timedelta(days = window_days - overlap_days)  # shift left by 210 days

        return windows[::-1]

In [ ]:
gts = GoogleTrendSyncer()

def sync(search_id):
    """
    fetch_fn(start, end) → returns list of {date, value}
    """
    def get_windows(search_id):
        past = gts.gtr.get_past_data(search_id)
        if past:
            start = datetime.fromtimestamp(past[-1].get('timestamp')) - timedelta(days=30)
        else:
            start = datetime.now().date() - timedelta(days = 4 * 365)
        end = datetime.now().date()

        windows = gts.generate_windows(start, end)
        return windows
    
    windows = get_windows(search_id)
    stitched = []

    for start, end in windows:
        chunk = fetch_fn(start, end)

        if not stitched:
            stitched = chunk
            continue

        overlap = self._find_overlap(stitched, chunk)
        scale = self.compute_scaling_factor(overlap)
        chunk = self.normalize_new_data(chunk, scale)
        stitched = self.merge_timeseries(stitched, chunk)

    self.upsert_timeseries(keyword, geo, stitched)

In [ ]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher
stf = SerpApiTrendsFetcher()
stf.initialize()

gtr = GoogleTrendsRepository()
searches_pending = gtr.searches_to_update()

current_search = searches_pending[0]
windows = sync(current_search.get('id'))

start, end = windows[0]
chunk = stf.fetch_chunk(keyword = current_search.get('search_keyword')
                        start = start, end)

In [ ]:
from gtrends.db.keywords import GoogleTrendsRepository

gtr = GoogleTrendsRepository()
past = gtr.get_past_data(1)

if not past:
    

# overlap = gts._find_overlap(past, new_data)
# scale = gts.compute_scaling_factor(overlap)
# normalized = gts.normalize_new_data(new_data, scale)
# merged = gts.merge_timeseries(past, normalized)

# gts.upsert_timeseries(keyword, geo, merged)



In [26]:
import os, sys
sys.path.insert(0, ".")
os.environ["SERPAPI_KEYS"] = "47cfd4d4c42c64c4050c883f7e8d43f93c63027b2d89fbc1f5b95ea9bbb801c2"


In [ ]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher

fetcher = SerpApiTrendsFetcher(geo="IN")

try:
    fetcher.initialize()
    data = fetcher.fetch_chunk(
    keyword="/g/11c265kd70",
    start="2024-01-01",
    end="2024-12-31",
    gprop="youtube"
    )
    print(f"success")
except Exception as e:
    print(f"error:{e}")

In [ ]:
import sys
import os
import time
from datetime import datetime, timedelta

# 1. SETUP: Ensure Python can see your 'gtrends' folder
# This adds the current directory to Python's search path
sys.path.append(os.getcwd())

# 2. IMPORTS: Bring in your classes
from gtrends.db.keywords import GoogleTrendsRepository
from gtrends.chunk.fetch import SerpApiTrendsFetcher
from gtrends.process.sync import GoogleTrendSyncer


def run_sync_pipeline():
    print("--- 🚀 Starting Google Trends Sync ---")

    # --- INITIALIZATION ---
    # 1. Connect to Database (Repository)
    repo = GoogleTrendsRepository()
    
    # 2. Setup Math Logic (Syncer)
    syncer = GoogleTrendSyncer()

    # 3. Setup API (Fetcher)
    fetcher = SerpApiTrendsFetcher()
    try:
        fetcher.initialize()
        print("✅ API Initialized")
    except Exception as e:
        print(f"❌ API Error: {e}")
        return

    # --- STEP 1: GET TASKS ---
    # Ask DB: "Which keywords need updating?"
    searches = repo.searches_to_update()
    
    if not searches:
        print("zzz No keywords pending update.")
        return

    print(f"📋 Found {len(searches)} keywords to process.")

    # --- STEP 2: PROCESS EACH KEYWORD ---
    for search in searches:
        # Extract details from the DB row
        search_id = search['id']
        keyword = search['search_keyword']
        geo = search.get('search_geo', 'IN')  # Default to India if missing
        gprop = search.get('gprop', 'youtube') # Default to youtube if missing

        print(f"\nProcessing: {keyword} ({geo})")

        # --- STEP 3: DETERMINE START DATE ---
        # Check if we have old data in the DB
        past_data = repo.get_past_data(search_id)
        
        end_date = datetime.now().date()
        
        if past_data:
            # HISTORY EXISTS: Start 30 days back from the last known date
            # (We need 30 days overlap to calculate the scaling factor)
            last_entry = past_data[-1]
            
            # Handle if 'timestamp' is stored as integer or datetime object
            if isinstance(last_entry['timestamp'], (int, float)):
                 last_date = datetime.fromtimestamp(last_entry['timestamp']).date()
            else:
                 last_date = last_entry['timestamp'] # Assuming it's already a date object

            start_date = last_date - timedelta(days=30)
            
            # Load existing data into our "stitched" list so we can append to it
            stitched_data = []
            for row in past_data:
                # Convert DB format to Syncer format: {'date': 'YYYY-MM-DD', 'value': 50}
                d_str = row['date'] if 'date' in row else datetime.fromtimestamp(row['timestamp']).strftime("%Y-%m-%d")
                stitched_data.append({"date": d_str, "value": row["value"]})
                
            print(f"   Refining existing data starting from: {start_date}")
        else:
            # NO HISTORY: Start fresh from 4 years ago
            start_date = end_date - timedelta(days=365*4)
            stitched_data = []
            print(f"   New keyword. Fetching fresh from: {start_date}")

        # --- STEP 4: GENERATE TIME WINDOWS ---
        # Break the long timeline into manageable 8-month chunks
        windows = syncer.generate_windows(start_date, end_date)
        print(f"   Created {len(windows)} time windows.")

        # --- STEP 5: FETCH & STITCH LOOP ---
        for start_str, end_str in windows:
            print(f"   Fetching {start_str} -> {end_str}...", end=" ")
            
            # A. FETCH from SerpApi
            try:
                chunk = fetcher.fetch_chunk(
                    keyword=keyword,
                    start=start_str,
                    end=end_str,
                    gprop=gprop
                )
            except Exception as e:
                print(f"[Error: {e}]")
                continue # Skip this chunk if API fails

            if not chunk:
                print("[Empty Data]")
                continue

            # B. STITCH (Merge)
            if not stitched_data:
                # If this is the very first piece of data, just save it
                stitched_data = chunk
                print("[First Chunk Saved]")
            else:
                # If we already have data, we must GLUE them together
                
                # 1. Find overlapping dates
                overlap = syncer._find_overlap(stitched_data, chunk)
                
                # 2. Calculate Math Ratio (Scaling Factor)
                scale = syncer.compute_scaling_factor(overlap)
                
                # 3. Adjust the new chunk to match the old scale
                normalized_chunk = syncer.normalize_new_data(chunk, scale)
                
                # 4. Merge them
                stitched_data = syncer.merge_timeseries(stitched_data, normalized_chunk)
                
                print(f"[Merged with scale: {scale:.2f}]")
            
            # Be polite to the API (wait 1 second)
            time.sleep(1)

        # --- STEP 6: SAVE TO DB ---
        if stitched_data:
            print(f"   💾 Saving {len(stitched_data)} rows to database...")
            syncer.upsert_timeseries(keyword, geo, stitched_data)
            print("   ✅ Done.")
        else:
            print("   ⚠️ No data found to save.")

    print("\n--- All jobs finished ---")

if __name__ == "__main__":
    run_sync_pipeline()

In [ ]:
import sys
import os
import time
from datetime import datetime, timedelta

# fetching function that fetches the data and scale the data and then stores in the db
from gtrends.apis.utils import SerpApiKeyManager
from gtrends.chunk.fetch import SerpApiTrendsFetcher
from gtrends.process.sync import GoogleTrendSyncer
from gtrends.db.keywords import GoogleTrendsRepository

key_manager = SerpApiKeyManager()
fetcher = SerpApiTrendsFetcher()
syncer = GoogleTrendSyncer()
gtr = GoogleTrendsRepository()

start = "01-01-2021"
end = datetime.today()

window_days = 240
overlap_days = 30

try:
        fetcher.initialize()
        print("✅ API Initialized")
except Exception as e:
        print(f"❌ API Error: {e}")
        # return

# getting the data to fetch from the db

search_list = []

if gtr.sync_search_ids():
        print("Ensured YOUTUBE entries exist for each exam")

search_list = gtr.searches_to_update()

if not search_list:
        print(f"No keywords pending for search!")
        # return

print(f"Found {len(search_list)} Keywords to process")


# fetch the data of keywoard 
# (self, keyword, start, end, gprop="youtube")
for data in search_list:
        try:
                search_id = data["id"]
                keyword = data["search_keyword"]
                gprop = data["search_platform"]
                geo = data["search_geo"]

                # global start and end dates 
                past_raw = gtr.get_past_data(search_id)
                past = syncer.normalise_past_data(past_raw) if past_raw else []
                
                if past_raw:
                        stitched = past
                        start = datetime.fromtimestamp(past_raw[-1]["timestamp"]) - timedelta(days = 30)

                else:
                        stitched = []
                        start = datetime.now().date() - timedelta(days = 4*365)

                end = datetime.now().date()

                #  Generate windows

                window_list = syncer.generate_windows(start,end,window_days,overlap_days)
                print(f"Created {len(window_list)} time windows.")
                
                fetched_data = []

                #  fetch window by window
                for win_start,win_end in window_list:
                        print(f"Fetching Data from {win_start} to {win_end}...")
                        try:
                                chunk = fetcher.fetch_chunk(
                                        keyword,
                                        win_start,
                                        win_end,
                                        gprop = gprop
                                )
                                if not chunk:
                                        print(f"empty chunk!")
                                        continue
                                if not stitched:
                                        stitched = sorted(chunk, key = lambda x: x["date"])
                                        continue
                        except Exception as e:
                                print(f"error fetching {e}")
                                continue
                        
                        print(f"Finding overlap!")
                        overlap = syncer._find_overlap(stitched,chunk)
                        if overlap:
                                scale = syncer.compute_scaling_factor(overlap)
                        else:
                                print(f"No overlap for {keyword} ({win_start}->{win_end}),using scale = 1.0")
                                scale = 1
                        
                        print(f"Syncing with scaling factor: {scale}")
                        normalized_chunk = syncer.normalize_new_data(chunk, scale)
                        stitched = syncer.merge_timeseries(stitched,normalized_chunk)
                
                        time.sleep(1)
                
                # save to DB

                if stitched :
                        print(f"Upserting {len(stitched)} data rows in DB")
                        syncer.upsert_timeseries(keyword, geo, stitched)
                        print(f"Done!")
                else:
                        print(f"No data to save!")

        except Exception as e:
                print(f"failed syncing {data.get('search_keyword')} : {e}")
                continue

# data is stored in the db